In [1]:
#Library Imports

import pandas as pd
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from gymnasium import spaces
from collections import deque

In [15]:
#Global Constants

TRAINING_ITERATIONS = 1
ACTION_MAP = {0:-1, 1:0, 2:1} #This is used to convert indices which need to be used in the replay buffer and tensor qnet into (Buy:-1, Hold:0, Sell:1) to work as a multiplier for the reward function

In [3]:
#Data Import 

dam = pd.read_parquet("dam_data_2026-01-01_2026-09-17.parquet")
dam["lmp"]

interval_start_utc
2026-01-01 00:00:00+00:00     35.06
2026-01-01 01:00:00+00:00     34.76
2026-01-01 02:00:00+00:00     33.11
2026-01-01 03:00:00+00:00     33.10
2026-01-01 04:00:00+00:00     34.49
                              ...  
2026-09-16 19:00:00+00:00     40.16
2026-09-16 20:00:00+00:00     45.98
2026-09-16 21:00:00+00:00     56.74
2026-09-16 22:00:00+00:00    100.99
2026-09-16 23:00:00+00:00    159.93
Name: lmp, Length: 6216, dtype: float64

In [4]:
# 1. Custom Gymnasium Environment
class BatteryEnv(gym.Env):
    def __init__(self):
        super().__init__()
        # State space: 25 dimension 
            # First 24 dimensions
                # Continuous vector from -infinity to infinity containing
                # Represents a 24 hour energy price schedule
                # This could be bounded if the training time is too high using a floor/ceiling. 
                # If the price is negative, you'll buy and if the price is really high, you'll sell. We're more interested in the middle range
            # 25th dimension
                # Capacity of the battery 

        low = np.append(np.full(24, -np.inf, dtype=np.float32), 0.0)
        high = np.append(np.full(24, np.inf, dtype=np.float32), 1.0)
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)
        # Action space: -1, 0, 1. Buy: -1, Hold: 0, Sell: 1. Represents the multiplier for the reward.  
            # Mapped by Action Map to get from {0,1,2} to {-1,0,1}
        self.action_space = spaces.Discrete(3)
        self.hours_passed = 0
        self.state = np.append(dam["lmp"][0:24], 0) #Append an empty battery capacity to the 24h schedule to form the state space

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.append(dam["lmp"][0:24],0) #Append an empty battery capacity to the 24h schedule to form the state space
        self.hours_passed = 0
        return self.state, {}

    def get_action_mask(self, state):
        capacity = state[24]
        mask = np.array([True, True, True], dtype=bool)
        if capacity == 0:
            mask[2] = False     #Disallow selling if the battery is empty
        elif capacity == 1:
            mask[0] = False     #Disallow buying if the battery is full
        else:
            print("Zack Warning: The capacity was neither 0 nor 1")
        return mask


    def step(self, action):
        current_price = self.state[0]
        #print(price_schedule)

        #print(action_schedule)
        actual_action = ACTION_MAP[action]   #Convert from action space {0,1,2} to reward multiplier {-1,0,1}
    
        #Calculate the reward of the decisions we made
        reward = actual_action * current_price
        #print(reward)


        self.hours_passed += 1
        if self.hours_passed + 24 >= len(dam["lmp"]):
            terminated = True   #End the episode when we've reached the end of the data
            truncated = False   #Truncated is only for artifical stoppages, which we will not have
        else:
            terminated = False   #Single step
            truncated = False

        old_capacity = self.state[24]
        new_capacity = old_capacity + actual_action
        self.state = np.append(dam["lmp"][self.hours_passed : self.hours_passed+24],new_capacity)   #Move the schedule to the next 24 forecast as well as update the battery capacity
        
        
        return self.state, reward, terminated, truncated, {}

In [5]:
# 2. Q-Network Architecture
class DQN(nn.Module):
    def __init__(self, state_dim=25, action_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, action_dim)
        )
        self.state_dim = state_dim
        self.action_dim = action_dim

    def forward(self, x):
        
        out = self.net(x)
        #print("forward 1: ", out)
        #out = out.view(-1, self.state_dim, self.action_dim)
        #print("forward 2: ", out)
        return out

In [6]:
# 3. Environment & Model Initialization
env = BatteryEnv()
q_net = DQN()
target_net = DQN()
target_net.load_state_dict(q_net.state_dict())

optimizer = optim.Adam(q_net.parameters(), lr=1e-3)
memory = deque(maxlen=5000)

gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.99
batch_size = 32

C:\Users\zacks\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\gymnasium\spaces\box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
C:\Users\zacks\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\gymnasium\spaces\box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [17]:
# 4. Training Loop
for episode in range(TRAINING_ITERATIONS):
    print(episode)
    if (episode % 25) == 0:
        print(f"Episode {episode}/{TRAINING_ITERATIONS}")
    state, _ = env.reset()
    done = False
    
    while not done:
        #print(env.hours_passed)
        action_mask = env.get_action_mask(state)
        valid_action_indices = np.where(action_mask)[0]

        # Epsilon-greedy action selection
        #print("Epsilon: ", epsilon)
        if random.random() < epsilon:
            action = np.random.choice(valid_action_indices)
            #print("type 1: ", action)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state)
                #print("state ", state_t)
                #print("qnet ", q_net(state_t))
                #print(q_net(state_t).shape)
                a = state_t
                #print("a: ", a)
                b = q_net(a)
                #print("b: ", b)
                c = b.argmax()
                #print("c: ", c)
                d = c.squeeze(0)
                #print("d: ", d)
                e = d.tolist()
                #print("e: ", e)
                action = e
                #print("type 2 ", action_)


        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        memory.append((state, action, reward, next_state, done))

        # Replay Buffer Sampling & Network Update
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            s_arr, a_arr, r_arr, ns_arr, d_arr = zip(*batch)

            states = torch.FloatTensor(np.array(s_arr))
            actions = torch.LongTensor(a_arr).unsqueeze(1)
            rewards = torch.FloatTensor(r_arr)
            next_states = torch.FloatTensor(np.array(ns_arr))
            dones = torch.FloatTensor(d_arr)

            # Compute current Q(s, a)
            z = states
            #print("z: ", z)
            y = q_net(z)
            #print("y: ", y)
            x = y.gather(1,actions)
            w = x.squeeze(1)
            q_values = w

            # Target Q-value using Bellman Equation
            with torch.no_grad():
                #print("target_shape: ", target_net(next_states).max(dim=1))
                #print("HELLLOOOOO")
                max_next_q = target_net(next_states).max(dim=1)[0]
                #print("rewards: ",rewards)
                #print("gamma: ", gamma)
                #print("max next q:", max_next_q)
                #print("dones: ", dones)
                targets = rewards + gamma * max_next_q * (1 - dones)

            loss = nn.MSELoss()(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    if episode % 20 == 0:
        target_net.load_state_dict(q_net.state_dict())

0
Episode 0/1


In [19]:
# 5. Verification Test
def test_input(state):
    q_net.eval()
    with torch.no_grad():
        inp = torch.FloatTensor(state).unsqueeze(0)
        q_vals = q_net(inp).numpy()[0]
        chosen_action = int(q_vals.argmax())
    return chosen_action, q_vals

print("Test: First 24 hours of 2026")
for hour in range(24):
    print("Selected Action: ", test_input(np.append(dam["lmp"][hour: hour + 24],0)))  # Expected: Unknown
    print("Price: ", dam["lmp"][hour])


Test: First 24 hours of 2026
Selected Action:  (1, array([367.3153  , 403.28442 ,   9.142402], dtype=float32))
Price:  35.06
Selected Action:  (1, array([386.08444 , 423.93628 ,   9.603095], dtype=float32))
Price:  34.76
Selected Action:  (1, array([396.34818, 435.31448,   9.86542], dtype=float32))
Price:  33.11
Selected Action:  (1, array([404.09372 , 443.85403 ,   9.578065], dtype=float32))
Price:  33.1
Selected Action:  (1, array([408.90097 , 449.1711  ,   9.033107], dtype=float32))
Price:  34.49
Selected Action:  (1, array([411.4321   , 451.9634   ,   8.9791975], dtype=float32))
Price:  30.84
Selected Action:  (1, array([408.00717 , 448.18423 ,   9.100854], dtype=float32))
Price:  30.15
Selected Action:  (1, array([408.72617 , 448.88467 ,   9.491794], dtype=float32))
Price:  33.19
Selected Action:  (1, array([404.92825, 444.5963 ,   9.6692 ], dtype=float32))
Price:  33.79
Selected Action:  (1, array([407.2634  , 447.08765 ,   9.866958], dtype=float32))
Price:  35.25
Selected Action

C:\Users\zacks\AppData\Local\Temp\ipykernel_28364\2663485567.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print("Price: ", dam["lmp"][hour])
